# Hydrogen Orbitals: Resonant Modes and 3-Phase Basis

This notebook explores hydrogen atom orbitals through the lens of:
1. **Resonant modes** - standing wave patterns analogous to reactive power oscillations
2. **3-phase power systems** - using symmetrical components as a natural basis
3. **Energy storage patterns** - probability density as distributed reactive energy

## Key Conceptual Framework

### Orbital → Resonant Mode Analogy
- **Radial nodes** (n-l-1): Like voltage standing wave patterns along a transmission line
- **Angular nodes** (l): Like phase relationships in multi-phase systems
- **Magnetic quantum number** (m): Directional bias, analogous to sequence components
- **Energy levels**: Resonant frequencies of the electromagnetic "cavity"

### Why 3-Phase is Natural
The spherical harmonics Y_l^m have natural 3-fold symmetry relationships:
- m = 0: Zero sequence (axially symmetric, real)
- m = ±1: Positive/negative sequence pairs (120° phase relationships)
- m = ±2: Higher harmonics (still phase-related)

The p-orbitals (l=1) with m = -1, 0, +1 form a perfect triplet that can be expressed
in a 3-phase rotating frame!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.special import sph_harm_y, genlaguerre, factorial
from matplotlib import cm
from matplotlib.colors import Normalize
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def R_nl(r, n, l, a0=1.0):
    """Hydrogen radial wavefunction R_{nl}(r)."""
    rho = 2.0 * r / (n * a0)
    L = genlaguerre(n - l - 1, 2*l + 1)(rho)
    pref = (2.0/(n*a0))**3
    norm = np.sqrt(pref * factorial(n - l - 1) / (2*n * factorial(n + l)))
    return norm * np.exp(-rho/2) * rho**l * L

def Y_lm(theta, phi, l, m):
    """Spherical harmonic Y_l^m(theta, phi)."""
    return sph_harm_y(m, l, phi, theta)

def psi_nlm(r, theta, phi, n, l, m, a0=1.0):
    """Full spatial wavefunction ψ_{nlm}(r,θ,φ)."""
    return R_nl(r, n, l, a0=a0) * Y_lm(theta, phi, l, m)

def density_nlm(r, theta, phi, n, l, m, a0=1.0):
    """Probability density |ψ|²."""
    psi = psi_nlm(r, theta, phi, n, l, m, a0=a0)
    return np.abs(psi)**2

## 1. Radial Wavefunctions: Standing Waves in Energy Storage

The radial part R_nl(r) shows **standing wave patterns** - regions where energy density
oscillates between potential and kinetic (like voltage and current in LC oscillations).

**Analogy to transmission lines:**
- Nodes in R_nl(r) ↔ voltage/current nodes on transmission line
- Antinodes ↔ maximum stored energy locations
- Different n values ↔ different resonant frequencies

In [ ]:
# Visualize radial wavefunctions as standing wave patterns
r = np.linspace(0, 30, 1000)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Radial Wavefunctions: Standing Wave Patterns (Resonant Modes)', fontsize=14, fontweight='bold')

orbitals = [
    (1, 0, '1s'),
    (2, 0, '2s'),
    (2, 1, '2p'),
    (3, 0, '3s'),
    (3, 1, '3p'),
    (3, 2, '3d')
]

for idx, (n, l, label) in enumerate(orbitals):
    ax = axes[idx // 3, idx % 3]
    
    # Radial wavefunction
    R = R_nl(r, n, l)
    
    # Probability density
    prob_density = r**2 * R**2
    
    ax.plot(r, R, 'b-', linewidth=2, label=f'R_{{{n}{l}}}')
    ax.fill_between(r, 0, R, alpha=0.3, color='blue')
    ax.axhline(0, color='k', linewidth=0.5, linestyle='--')
    
    # Mark nodes
    nodes = n - l - 1
    ax.set_title(f'{label}: {nodes} radial node(s)\n(n={n}, l={l})', fontweight='bold')
    ax.set_xlabel('r (Bohr radii)')
    ax.set_ylabel('R(r)')
    ax.grid(True, alpha=0.3)
    ax.legend()
    
    # Add text annotation about energy storage
    ax.text(0.95, 0.95, f'E = -13.6/{n}² eV', 
            transform=ax.transAxes, ha='right', va='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

# Now plot radial probability densities
fig, ax = plt.subplots(figsize=(12, 6))
ax.set_title('Radial Probability Density: Energy Distribution (like reactive power)', fontsize=14, fontweight='bold')

for n, l, label in orbitals:
    R = R_nl(r, n, l)
    prob_density = r**2 * R**2
    ax.plot(r, prob_density, linewidth=2, label=label)

ax.set_xlabel('r (Bohr radii)', fontsize=12)
ax.set_ylabel('4πr² |R(r)|²', fontsize=12)
ax.legend(fontsize=10, ncol=2)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 30)
plt.tight_layout()
plt.show()

## 2. Angular Structure: Phase Relationships

The spherical harmonics Y_l^m encode **angular phase patterns**.

**Key insight for 3-phase analogy:**
- m = 0: **Zero sequence** - no rotation, axially symmetric
- m = ±1: **Positive/negative sequence** - rotating patterns, 120° symmetry
- Complex exponential e^(imφ) ↔ rotating phasor e^(jωt)

In [ ]:
# Visualize spherical harmonics in 2D polar plots
theta = np.linspace(0, np.pi, 100)
phi = np.linspace(0, 2*np.pi, 100)
THETA, PHI = np.meshgrid(theta, phi)

fig, axes = plt.subplots(2, 3, figsize=(15, 10), subplot_kw={'projection': 'polar'})
fig.suptitle('Spherical Harmonics: Angular Phase Patterns', fontsize=14, fontweight='bold')

configs = [
    (1, -1, 'p: m=-1 (negative seq)'),
    (1, 0, 'p: m=0 (zero seq)'),
    (1, 1, 'p: m=+1 (positive seq)'),
    (2, -1, 'd: m=-1'),
    (2, 0, 'd: m=0'),
    (2, 2, 'd: m=+2')
]

for idx, (l, m, label) in enumerate(configs):
    ax = axes[idx // 3, idx % 3]
    
    # Plot at theta = π/2 (equatorial plane)
    phi_vals = np.linspace(0, 2*np.pi, 200)
    Y = Y_lm(np.pi/2, phi_vals, l, m)
    magnitude = np.abs(Y)**2
    
    ax.plot(phi_vals, magnitude, linewidth=2)
    ax.fill(phi_vals, magnitude, alpha=0.3)
    ax.set_title(label, fontweight='bold', pad=20)
    ax.grid(True, alpha=0.3)
    
    # Add phase annotation for m≠0
    if m != 0:
        phase_text = f'{abs(m)} × φ rotation'
        ax.text(0.5, 0.95, phase_text, transform=ax.transAxes,
                ha='center', va='top', fontsize=9,
                bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))

plt.tight_layout()
plt.show()

## 3. The 3-Phase Power Connection: p-Orbitals as Balanced System

The three p-orbitals (px, py, pz) form a **balanced 3-phase system**!

### Mathematical Connection:
The spherical harmonic p-orbitals with m = -1, 0, +1 can be transformed to:
- pz = Y₁⁰ (real, zero sequence)
- px = (Y₁⁻¹ - Y₁⁺¹)/√2 (real combination)
- py = i(Y₁⁻¹ + Y₁⁺¹)/√2 (real combination)

These form an **orthogonal triad** at 90° (not 120°), but the underlying complex basis
Y₁⁻¹, Y₁⁺¹ represents **counter-rotating components** - exactly like positive and
negative sequence in 3-phase power!

### Clarke/Park Transform Analogy:
- Complex spherical harmonics e^(imφ) ↔ sequence components (rotating phasors)
- Real cartesian orbitals (px, py, pz) ↔ phase quantities (a, b, c)
- Transformation between them ↔ Clarke transform (abc → αβ0)

In [ ]:
# Construct real p-orbitals from complex spherical harmonics
def p_orbital_cartesian(r, theta, phi, orbital_type, a0=1.0):
    """Real p-orbitals in Cartesian representation.
    
    orbital_type: 'px', 'py', or 'pz'
    These are linear combinations of Y_1^m.
    """
    R = R_nl(r, n=2, l=1, a0=a0)
    
    if orbital_type == 'pz':
        # pz is just Y_1^0 (real, m=0)
        return R * Y_lm(theta, phi, 1, 0)
    
    elif orbital_type == 'px':
        # px = (Y_1^-1 - Y_1^+1) / sqrt(2)
        Y_m1 = Y_lm(theta, phi, 1, -1)
        Y_p1 = Y_lm(theta, phi, 1, 1)
        return R * (Y_m1 - Y_p1) / np.sqrt(2)
    
    elif orbital_type == 'py':
        # py = i(Y_1^-1 + Y_1^+1) / sqrt(2)
        Y_m1 = Y_lm(theta, phi, 1, -1)
        Y_p1 = Y_lm(theta, phi, 1, 1)
        return R * 1j * (Y_m1 + Y_p1) / np.sqrt(2)
    
    else:
        raise ValueError("orbital_type must be 'px', 'py', or 'pz'")

# Visualize the 3-phase triad
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('p-Orbitals: A Balanced 3-Phase System', fontsize=14, fontweight='bold')

extent = 10
x = np.linspace(-extent, extent, 150)
z = np.linspace(-extent, extent, 150)
X, Z = np.meshgrid(x, z)
Y_plane = 0  # xz plane

# Convert to spherical
R_grid = np.sqrt(X**2 + Y_plane**2 + Z**2)
THETA_grid = np.arccos(Z / (R_grid + 1e-10))
PHI_grid = np.arctan2(Y_plane, X)

for idx, orbital in enumerate(['px', 'py', 'pz']):
    ax = axes[idx]
    
    psi = p_orbital_cartesian(R_grid, THETA_grid, PHI_grid, orbital)
    density = np.real(psi * np.conj(psi))
    
    im = ax.contourf(X, Z, density, levels=30, cmap='RdBu_r')
    ax.contour(X, Z, density, levels=10, colors='black', linewidths=0.5, alpha=0.3)
    
    ax.set_xlabel('x (Bohr radii)')
    ax.set_ylabel('z (Bohr radii)' if idx == 0 else '')
    ax.set_title(f'{orbital} orbital\n({"Phase A" if orbital=="px" else "Phase B" if orbital=="py" else "Phase C"})',
                 fontweight='bold')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)
    
    plt.colorbar(im, ax=ax, label='Density')

plt.tight_layout()
plt.show()

print("\n=== 3-Phase Power System Analogy ===")
print("px, py, pz form orthogonal basis (like abc phases in balanced system)")
print("Y_1^-1, Y_1^0, Y_1^+1 are sequence components (negative, zero, positive)")
print("Transformation between them is analogous to Clarke transform!")

## 4. Sequence Component Representation

Let's explicitly show how complex spherical harmonics relate to sequence components.

For l=1 (p-orbitals):
- **Y₁⁺¹** ~ positive sequence (rotates as e^(+iφ))
- **Y₁⁰** ~ zero sequence (no rotation)
- **Y₁⁻¹** ~ negative sequence (rotates as e^(-iφ))

Just like in 3-phase power:
- Positive sequence: A·e^(j0°), A·e^(j240°), A·e^(j120°)
- Negative sequence: A·e^(j0°), A·e^(j120°), A·e^(j240°)
- Zero sequence: A, A, A (no phase shift)

In [ ]:
# Visualize sequence components as functions of phi angle
phi_vals = np.linspace(0, 2*np.pi, 200)
theta_fixed = np.pi/2  # Equatorial plane

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
fig.suptitle('Sequence Component Analysis: Y_l^m as Rotating Phasors', fontsize=14, fontweight='bold')

for l_idx, l in enumerate([1, 2]):
    # Positive sequence (m = +l)
    ax = axes[0, l_idx]
    Y_pos = Y_lm(theta_fixed, phi_vals, l, l)
    ax.plot(phi_vals, np.real(Y_pos), 'r-', linewidth=2, label='Real part')
    ax.plot(phi_vals, np.imag(Y_pos), 'b--', linewidth=2, label='Imag part')
    ax.set_title(f'Positive Sequence: Y_{{{l}}}^{{+{l}}} ∝ e^{{+i{l}φ}}', fontweight='bold')
    ax.set_xlabel('φ (radians)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='k', linewidth=0.5)
    
    # Zero sequence (m = 0)
    ax = axes[1, l_idx]
    Y_zero = Y_lm(theta_fixed, phi_vals, l, 0)
    ax.plot(phi_vals, np.real(Y_zero), 'g-', linewidth=2, label='Real (constant)')
    ax.plot(phi_vals, np.imag(Y_zero), 'g--', linewidth=2, label='Imag (zero)', alpha=0.5)
    ax.set_title(f'Zero Sequence: Y_{{{l}}}^{{0}} (no φ dependence)', fontweight='bold')
    ax.set_xlabel('φ (radians)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='k', linewidth=0.5)
    
    # Negative sequence (m = -l)
    ax = axes[2, l_idx]
    Y_neg = Y_lm(theta_fixed, phi_vals, l, -l)
    ax.plot(phi_vals, np.real(Y_neg), 'r-', linewidth=2, label='Real part')
    ax.plot(phi_vals, np.imag(Y_neg), 'b--', linewidth=2, label='Imag part')
    ax.set_title(f'Negative Sequence: Y_{{{l}}}^{{-{l}}} ∝ e^{{-i{l}φ}}', fontweight='bold')
    ax.set_xlabel('φ (radians)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='k', linewidth=0.5)

axes[0, 0].text(0.02, 0.98, 'l=1 (p-orbitals)', transform=axes[0, 0].transAxes,
                fontsize=12, verticalalignment='top', fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))
axes[0, 1].text(0.02, 0.98, 'l=2 (d-orbitals)', transform=axes[0, 1].transAxes,
                fontsize=12, verticalalignment='top', fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))

plt.tight_layout()
plt.show()

## 5. Reactive Power Interpretation: Energy Storage Oscillations

**Key conceptual leap:** The probability density |ψ|² represents the spatial distribution
of **stored electromagnetic energy** - analogous to reactive power in AC circuits.

### Resonant Mode Picture:
1. **Ground state (1s)**: Fundamental mode, lowest frequency, maximum energy storage near nucleus
2. **Excited states**: Higher harmonics with nodes (like overtones on a string)
3. **Angular momentum states**: Different phase patterns around the axis

### Energy Oscillation:
While our static plots show |ψ|², in reality there's time evolution:
- ψ(r,t) = ψ(r)·e^(-iEt/ℏ)
- This is exactly like a phasor: V(t) = V·e^(jωt)
- Energy oscillates between kinetic and potential (like L and C in LC tank)

In [ ]:
# Create 3D isosurface visualization showing "reactive power" distribution
from mpl_toolkits.mplot3d import Axes3D

def plot_3d_isosurface(n, l, m, threshold=0.01):
    """Plot 3D isosurface of probability density."""
    # Create 3D grid
    extent = n**2 * 8  # Scale with quantum number
    res = 50
    x = np.linspace(-extent, extent, res)
    y = np.linspace(-extent, extent, res)
    z = np.linspace(-extent, extent, res)
    X, Y, Z = np.meshgrid(x, y, z)
    
    # Convert to spherical
    R = np.sqrt(X**2 + Y**2 + Z**2)
    THETA = np.arccos(Z / (R + 1e-10))
    PHI = np.arctan2(Y, X)
    
    # Calculate density
    density = density_nlm(R, THETA, PHI, n, l, m)
    
    # Create figure
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    # Plot isosurface
    from skimage import measure
    try:
        verts, faces, normals, values = measure.marching_cubes(density, threshold)
        # Scale vertices back to physical coordinates
        verts = verts * (2*extent/res) - extent
        
        ax.plot_trisurf(verts[:, 0], verts[:, 1], faces, verts[:, 2],
                        cmap='viridis', alpha=0.7, linewidth=0)
    except:
        print(f"Could not generate isosurface for n={n}, l={l}, m={m}")
        return
    
    ax.set_xlabel('x (Bohr radii)')
    ax.set_ylabel('y (Bohr radii)')
    ax.set_zlabel('z (Bohr radii)')
    ax.set_title(f'3D Energy Density: n={n}, l={l}, m={m}\n'
                 f'(Resonant Mode with E = -13.6/{n}² eV)',
                 fontweight='bold')
    
    # Set equal aspect ratio
    max_range = extent
    ax.set_xlim([-max_range, max_range])
    ax.set_ylim([-max_range, max_range])
    ax.set_zlim([-max_range, max_range])
    
    plt.tight_layout()
    plt.show()

# Visualize several orbitals as 3D energy distributions
print("Generating 3D isosurfaces (this may take a moment)...\n")
try:
    from skimage import measure
    
    plot_3d_isosurface(2, 1, 0, threshold=0.0001)  # 2pz
    plot_3d_isosurface(3, 2, 0, threshold=0.00005)  # 3dz²
    
except ImportError:
    print("scikit-image not available. Skipping 3D isosurface plots.")
    print("Install with: pip install scikit-image --break-system-packages")

## 6. Summary: Key Conceptual Bridges

### Hydrogen Atom ↔ 3-Phase Power System

| Quantum Concept | Power System Analog |
|----------------|---------------------|
| Orbital ψ_nlm | Resonant mode in EM cavity |
| Energy level E_n | Resonant frequency ω |
| Probability density |ψ|² | Stored reactive energy distribution |
| Radial nodes | Voltage standing wave nodes |
| Spherical harmonic Y_l^m | Phasor/sequence component |
| m = +1 | Positive sequence (CCW rotation) |
| m = 0 | Zero sequence (no rotation) |
| m = -1 | Negative sequence (CW rotation) |
| px, py, pz basis | abc phase quantities |
| Y_-1, Y_0, Y_+1 basis | αβ0 sequence components |
| Time evolution e^(-iEt/ℏ) | Phasor rotation e^(jωt) |

### Why This Works:
Both systems involve:
1. **Standing wave patterns** in bounded domains
2. **Complex exponential representations** (phasors vs quantum phases)
3. **Orthogonal decomposition** (symmetrical components vs quantum numbers)
4. **Energy storage oscillations** (reactive power vs quantum probability)
5. **Rotation/phase relationships** (sequence rotation vs angular momentum)

### Deeper Insight:
The mathematical structure of 3-phase power (balanced systems with symmetrical components)
is *isomorphic* to the angular structure of quantum wavefunctions! The positive/negative
sequence pairs are exactly analogous to the m = ±1 pairs in spherical harmonics.

This isn't just an analogy - it's a fundamental property of **SO(3) symmetry** appearing
in both contexts:
- Rotations in 3D space (quantum mechanics)
- Balanced 3-phase systems (power engineering)

### For Further Exploration:
1. Try implementing a "Clarke transform" for converting between real (px,py,pz) and complex (Y_-1,Y_0,Y_+1) bases
2. Explore d-orbitals (l=2) as "5-phase systems" or higher harmonic content
3. Calculate "sequence power flow" by looking at cross-terms between different m values
4. Investigate time-dependent superpositions as beating modes (like unbalanced 3-phase)

## Bonus: Building Orbitals from 3-Phase Components

Let's explicitly construct the real p-orbitals from the complex sequence components,
showing the transformation is exactly like Clarke/Park transforms in power systems.

In [ ]:
# Demonstrate the "Clarke Transform" for p-orbitals
print("=== Orbital Transform (analogous to Clarke Transform) ===")
print("\nComplex basis (sequence components):")
print("  Y_1^(+1) : positive sequence (rotates +φ)")
print("  Y_1^(0)  : zero sequence (no rotation)")
print("  Y_1^(-1) : negative sequence (rotates -φ)")
print("\nReal basis (Cartesian/phase quantities):")
print("  px = (Y_1^(-1) - Y_1^(+1)) / √2")
print("  py = i(Y_1^(-1) + Y_1^(+1)) / √2")
print("  pz = Y_1^(0)")
print("\nThis is exactly analogous to:")
print("  Power sequence components (α⁺, 0, α⁻) ↔ Phase quantities (a, b, c)")
print("\nTransformation matrix structure:")
print("  [px]   [  1/√2    0    -1/√2  ] [Y_+1]")
print("  [py] = [  i/√2    0     i/√2  ] [Y_0 ]")
print("  [pz]   [   0      1      0    ] [Y_-1]")

# Verify orthogonality
print("\n=== Orthogonality Check ===")
print("p-orbitals are orthonormal (like balanced 3-phase):")

# Sample points
n_samples = 1000
r_sample = np.random.exponential(2, n_samples)  # Random radii
theta_sample = np.arccos(2*np.random.rand(n_samples) - 1)
phi_sample = 2*np.pi*np.random.rand(n_samples)

# Calculate orbitals
px_vals = p_orbital_cartesian(r_sample, theta_sample, phi_sample, 'px')
py_vals = p_orbital_cartesian(r_sample, theta_sample, phi_sample, 'py')
pz_vals = p_orbital_cartesian(r_sample, theta_sample, phi_sample, 'pz')

# Inner products (approximately, via sampling)
vol_element = r_sample**2 * np.sin(theta_sample)

inner_px_py = np.sum(np.real(px_vals * np.conj(py_vals)) * vol_element) / n_samples
inner_px_pz = np.sum(np.real(px_vals * np.conj(pz_vals)) * vol_element) / n_samples
inner_py_pz = np.sum(np.real(py_vals * np.conj(pz_vals)) * vol_element) / n_samples

print(f"  <px|py> ≈ {inner_px_py:.6f} (should be ~0)")
print(f"  <px|pz> ≈ {inner_px_pz:.6f} (should be ~0)")
print(f"  <py|pz> ≈ {inner_py_pz:.6f} (should be ~0)")
print("\nJust like 3-phase abc components are orthogonal in balanced system!")